In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist

# # ۱. تنظیمات مسیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_filename = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_deviation_monitoring_output1.xlsx'

# all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
#                 'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
#                 'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# # پارامترهای زمانی (پنجره‌ها)
# smoothing_span = 5     # برای اسموت کردن ورودی (نویزگیری)
# win_48h = 48           # بازه ۲ روزه
# win_7d = 168           # بازه ۷ روزه

# # ۲. خواندن داده‌ها
# df_raw = pd.read_excel(file_path)
# df_raw['date'] = pd.to_datetime(df_raw['date'])
# df_raw = df_raw.sort_values(by='date')

# # ۳. حذف ۱۰ درصد داده‌های پرت (DBSCAN)
# data_values = df_raw[all_features].values
# dbscan = DBSCAN(eps=5, min_samples=10)
# labels = dbscan.fit_predict(data_values)
# cluster_centers = {cid: data_values[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def get_dist(i):
#     if labels[i] != -1:
#         return cdist(data_values[i].reshape(1, -1), cluster_centers[labels[i]].reshape(1, -1))[0][0]
#     all_c = np.array(list(cluster_centers.values()))
#     return np.min(cdist(data_values[i].reshape(1, -1), all_c)) if len(all_c) > 0 else 0

# df_raw['distance'] = [get_dist(i) for i in range(len(df_raw))]
# num_to_remove = int(len(df_raw)*0.10)
# df_cleaned = df_raw.sort_values(by='distance').iloc[:(len(df_raw)-num_to_remove)].copy()
# df_cleaned = df_cleaned.sort_values(by='date')

# # ۴. محاسبه هر دو بازه زمانی
# print("در حال محاسبه پیش‌بینی‌های ۴۸ ساعته و ۷ روزه...")

# for sensor in all_features:
#     # الف) ورودی اسموت شده
#     df_cleaned[f'{sensor}_smooth'] = df_cleaned[sensor].ewm(span=smoothing_span, adjust=False).mean()

#     # ب) پیش‌بینی ۴۸ ساعته (روند کوتاه مدت)
#     df_cleaned[f'pred_48h_{sensor.split("_")[1]}'] = df_cleaned[sensor].rolling(window=win_48h).mean()

#     # ج) پیش‌بینی ۷ روزه (روند میان مدت)
#     df_cleaned[f'pred_7d_{sensor.split("_")[1]}'] = df_cleaned[sensor].rolling(window=win_7d).mean()

#     # د) تشخیص انحراف (مبنا را روی ۷ روزه می‌گذاریم که باثبات‌تر است)
#     rolling_std_7d = df_cleaned[sensor].rolling(window=win_7d).std()
#     upper = df_cleaned[f'pred_7d_{sensor.split("_")[1]}'] + (3*rolling_std_7d)
#     lower = df_cleaned[f'pred_7d_{sensor.split("_")[1]}'] - (3*rolling_std_7d)

#     df_cleaned[f'is_dev_{sensor.split("_")[1]}'] = ((df_cleaned[f'{sensor}_smooth'] > upper) | (df_cleaned[f'{sensor}_smooth'] < lower)).astype(int)

# # ۵. فیلتر ۳۰ روز آخر
# last_date = df_cleaned['date'].max()
# df_final = df_cleaned[df_cleaned['date'] >= (last_date - pd.Timedelta(days=30))].copy()

# # ۶. مرتب‌سازی ستون‌ها برای خروجی اکسل
# smooth_cols = [f'{s}_smooth' for s in all_features]

# pred_48h_cols = [f'pred_48h_{s.split("_")[1]}' for s in target_sensors]
# pred_7d_cols = [f'pred_7d_{s.split("_")[1]}' for s in target_sensors]
# dev_cols = [f'is_dev_{s.split("_")[1]}' for s in target_sensors]

# final_columns = ['date'] + all_features + smooth_cols + pred_48h_cols + pred_7d_cols + dev_cols

# # ذخیره نهایی
# df_final[final_columns].to_excel(output_filename, index=False)
# print(f"فایل با موفقیت ذخیره شد. شامل هر دو بازه ۴۸ ساعته و ۷ روزه.")

In [ ]:
# import pandas as pd
# import numpy as np
# from sklearn.cluster import DBSCAN
# from scipy.spatial.distance import cdist

# # ۱. تنظیمات مسیرها
# file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
# output_wide = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_deviation_monitoring_output1.xlsx'
# output_long = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_deviation_monitoring_output2.xlsx'

# all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
#                 'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
#                 'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

# target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# # پارامترهای زمانی
# smoothing_span = 5
# win_48h = 48
# win_7d = 168

# # ۲. خواندن و پاکسازی داده‌ها (DBSCAN)
# df_raw = pd.read_excel(file_path)
# df_raw['date'] = pd.to_datetime(df_raw['date'])
# df_raw = df_raw.sort_values(by='date')

# data_values = df_raw[all_features].values
# dbscan = DBSCAN(eps=5, min_samples=10)
# labels = dbscan.fit_predict(data_values)
# cluster_centers = {cid: data_values[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

# def get_dist(i):
#     if labels[i] != -1:
#         return cdist(data_values[i].reshape(1, -1), cluster_centers[labels[i]].reshape(1, -1))[0][0]
#     all_c = np.array(list(cluster_centers.values()))
#     return np.min(cdist(data_values[i].reshape(1, -1), all_c)) if len(all_c) > 0 else 0

# df_raw['distance'] = [get_dist(i) for i in range(len(df_raw))]

# # --- اصلاح خط ۳۹ ---
# num_to_remove = int(len(df_raw) * 0.10) 
# # ------------------

# df_cleaned = df_raw.sort_values(by='distance').iloc[:(len(df_raw)-num_to_remove)].copy()
# df_cleaned = df_cleaned.sort_values(by='date')

# # ۳. محاسبات شاخص‌ها
# for sensor in all_features:
#     # اسموت کردن
#     df_cleaned[f'{sensor}_smoothed'] = df_cleaned[sensor].ewm(span=smoothing_span, adjust=False).mean()
#     # پیش‌بینی ۴۸ ساعته
#     df_cleaned[f'pred_48h_{sensor}'] = df_cleaned[sensor].rolling(window=win_48h).mean()
#     # پیش‌بینی ۷ روزه
#     df_cleaned[f'pred_7d_{sensor}'] = df_cleaned[sensor].rolling(window=win_7d).mean()

#     # تشخیص انحراف (اصلاح ضرب در ۳)
#     rstd = df_cleaned[sensor].rolling(window=win_7d).std()
#     upper = df_cleaned[f'pred_7d_{sensor}'] + (3 * rstd)
#     lower = df_cleaned[f'pred_7d_{sensor}'] - (3 * rstd)
#     df_cleaned[f'is_dev_{sensor}'] = ((df_cleaned[f'{sensor}_smoothed'] > upper) | (df_cleaned[f'{sensor}_smoothed'] < lower)).astype(int)

# # ۴. فیلتر ۳۰ روز آخر
# last_date = df_cleaned['date'].max()
# df_final = df_cleaned[df_cleaned['date'] >= (last_date - pd.Timedelta(days=30))].copy()

# # ۵. تولید خروجی اول (Wide Structure)
# df_final.to_excel(output_wide, index=False)
# print(f"خروجی اول ذخیره شد.")

# # ۶. تولید خروجی دوم (Long Structure)
# long_data_list = []
# for sensor in target_sensors:
#     temp_df = pd.DataFrame()
#     temp_df['date'] = df_final['date']
#     temp_df['AssetID'] = sensor
#     temp_df['AssetID_smoothed'] = df_final[f'{sensor}_smoothed']
#     temp_df['predicted_48'] = df_final[f'pred_48h_{sensor}']
#     temp_df['predicted_1D'] = df_final[f'pred_7d_{sensor}']
#     temp_df['is_dev'] = df_final[f'is_dev_{sensor}']
#     long_data_list.append(temp_df)

# df_long_final = pd.concat(long_data_list, ignore_index=True)

# df_long_final = df_long_final.dropna(subset=['predicted_1D'])

# df_long_final.to_excel(output_long, index=False)
# print(f"خروجی دوم با استراکچر جدید ایجاد شد.")

خروجی اول ذخیره شد.
خروجی دوم با استراکچر جدید ایجاد شد.


In [3]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist

# ۱. تنظیمات مسیرها
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_wide = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_deviation_monitoring_output1.xlsx'
output_long = r'outputs\G11\dsas_g11_bearings_vibration_temp_deviation_monitoring\deviation_monitoring\dsas_g11_deviation_monitoring_output2.xlsx'

all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
                'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# پارامترهای زمانی
smoothing_span = 5
win_48h = 48
win_7d = 168

# ۲. خواندن و پاکسازی داده‌ها (DBSCAN)
df_raw = pd.read_excel(file_path)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values(by='date')

data_values = df_raw[all_features].values
dbscan = DBSCAN(eps=5, min_samples=10)
labels = dbscan.fit_predict(data_values)
cluster_centers = {cid: data_values[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

def get_dist(i):
    if labels[i] != -1:
        return cdist(data_values[i].reshape(1, -1), cluster_centers[labels[i]].reshape(1, -1))[0][0]
    all_c = np.array(list(cluster_centers.values()))
    return np.min(cdist(data_values[i].reshape(1, -1), all_c)) if len(all_c) > 0 else 0

df_raw['distance'] = [get_dist(i) for i in range(len(df_raw))]

# --- اصلاح خط ۳۹ ---
num_to_remove = int(len(df_raw) * 0.10) 
# ------------------

df_cleaned = df_raw.sort_values(by='distance').iloc[:(len(df_raw)-num_to_remove)].copy()
df_cleaned = df_cleaned.sort_values(by='date')

# ۳. محاسبات شاخص‌ها
for sensor in all_features:
    # اسموت کردن
    df_cleaned[f'{sensor}_smoothed'] = df_cleaned[sensor].ewm(span=smoothing_span, adjust=False).mean()
    # پیش‌بینی ۴۸ ساعته
    df_cleaned[f'pred_48h_{sensor}'] = df_cleaned[sensor].rolling(window=win_48h).mean()
    # پیش‌بینی ۷ روزه
    df_cleaned[f'pred_7d_{sensor}'] = df_cleaned[sensor].rolling(window=win_7d).mean()

    # تشخیص انحراف (اصلاح ضرب در ۳)
    rstd = df_cleaned[sensor].rolling(window=win_7d).std()
    upper = df_cleaned[f'pred_7d_{sensor}'] + (3 * rstd)
    lower = df_cleaned[f'pred_7d_{sensor}'] - (3 * rstd)
    df_cleaned[f'is_dev_{sensor}'] = ((df_cleaned[f'{sensor}_smoothed'] > upper) | (df_cleaned[f'{sensor}_smoothed'] < lower)).astype(int)

# ۴. فیلتر ۳۰ روز آخر
last_date = df_cleaned['date'].max()
df_final = df_cleaned[df_cleaned['date'] >= (last_date - pd.Timedelta(days=30))].copy()

# ۵. تولید خروجی اول (Wide Structure) - با تغییر ترتیب ستون‌ها
# گرفتن لیست تمام ستون‌ها به جز date
other_columns = [col for col in df_final.columns if col != 'date']
# ایجاد ترتیب جدید با date در ابتدا
new_column_order = ['date'] + other_columns
# اعمال ترتیب جدید
df_final = df_final[new_column_order]

df_final.to_excel(output_wide, index=False)
print(f"خروجی اول ذخیره شد. (ستون date در ابتدا قرار گرفت)")

# ۶. تولید خروجی دوم (Long Structure)
long_data_list = []
for sensor in target_sensors:
    temp_df = pd.DataFrame()
    temp_df['date'] = df_final['date']
    temp_df['AssetID'] = sensor
    temp_df['AssetID_smoothed'] = df_final[f'{sensor}_smoothed']
    temp_df['predicted_48'] = df_final[f'pred_48h_{sensor}']
    temp_df['predicted_1D'] = df_final[f'pred_7d_{sensor}']
    temp_df['is_dev'] = df_final[f'is_dev_{sensor}']
    long_data_list.append(temp_df)

df_long_final = pd.concat(long_data_list, ignore_index=True)

df_long_final = df_long_final.dropna(subset=['predicted_1D'])

df_long_final.to_excel(output_long, index=False)
print(f"خروجی دوم با استراکچر جدید ایجاد شد.")

خروجی اول ذخیره شد. (ستون date در ابتدا قرار گرفت)
خروجی دوم با استراکچر جدید ایجاد شد.
